# Participantes:
Luiz Henrique Barbosa - 2023121494, 
Guilherme Muniz - 2023121285, 
Gabriel Gripp - 202312139, 
Jorge Luiz Barbosa - 2023121496

In [24]:
import os
import time
import random
import copy
from typing import List, Tuple


In [25]:
# COLOQUE O CAMINHO DA PASTA AQUI 
PASTA = "low_dimensional"
# PASTA = r"C:\Users\SeuNome\Downloads\low_dimensional"  # Windows
# PASTA = "/home/seunome/Downloads/low_dimensional"      # Linux/Mac

def read_instance(filepath: str) -> Tuple[int, int, List[Tuple[float, float]]]:
    """Lê instância: retorna (n_itens, capacidade, [(peso, valor), ...])"""
    with open(filepath, 'r') as f:
        lines = [l.strip() for l in f if l.strip()]
    n, cap = map(float, lines[0].split(','))
    items = []
    for line in lines[1:]:
        w, v = map(float, line.split(','))
        items.append((w, v))
    return int(n), cap, items

In [26]:
# 1. ALGORITMO EXATO – Programação Dinâmica. Funciona com pesos inteiros; pesos flutuantes são escalados.

def exato_PD(capacity: float, items: List[Tuple[float, float]]) -> Tuple[float, List[int]]:
    """
    Programação Dinâmica 0/1 com suporte a pesos flutuantes via escalonamento.
    Retorna (melhor_valor, lista_de_indices_selecionados).
    """
    # Verificar se pesos são inteiros ou flutuantes
    all_weights = [items[i][0] for i in range(len(items))]
    is_float = any('.' in str(w) for w in all_weights)

    if is_float:
        # Escalar para inteiros com 3 casas decimais
        scale = 1000
        W = int(round(capacity * scale))
        weights = [int(round(w * scale)) for w, _ in items]
    else:
        scale = 1
        W = int(capacity)
        weights = [int(w) for w, _ in items]

    values = [v for _, v in items]
    n = len(items)

    # Tabela DP
    dp = [[0.0] * (W + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        wi = weights[i - 1]
        vi = values[i - 1]
        for w in range(W + 1):
            dp[i][w] = dp[i - 1][w]
            if wi <= w and dp[i - 1][w - wi] + vi > dp[i][w]:
                dp[i][w] = dp[i - 1][w - wi] + vi

    # Rastrear itens selecionados
    selected = []
    w = W
    for i in range(n, 0, -1):
        if dp[i][w] != dp[i - 1][w]:
            selected.append(i - 1)
            w -= weights[i - 1]

    return dp[n][W], selected



In [27]:
# 2. FIRST FIT – insere o item na primeira posição que couber

def first_fit(capacity: float, items: List[Tuple[float, float]]) -> Tuple[float, List[int]]:
    """
    Percorre os itens em ordem; aceita o item se couber na capacidade restante.
    """
    remaining = capacity
    total_value = 0.0
    selected = []
    for i, (w, v) in enumerate(items):
        if w <= remaining:
            selected.append(i)
            remaining -= w
            total_value += v
    return total_value, selected

In [28]:
# 3. BEST FIT – ordena por razão valor/peso (melhor custo-benefício primeiro)
def best_fit(capacity: float, items: List[Tuple[float, float]]) -> Tuple[float, List[int]]:
    """
    Ordena itens pela razão valor/peso (decrescente) e insere os que couberem.
    """
    ratios = []
    for i, (w, v) in enumerate(items):
        ratio = v / w if w > 0 else float('inf')
        ratios.append((ratio, i, w, v))
    ratios.sort(reverse=True)

    remaining = capacity
    total_value = 0.0
    selected = []
    for _, i, w, v in ratios:
        if w <= remaining:
            selected.append(i)
            remaining -= w
            total_value += v
    return total_value, selected


In [29]:
# 4. WORST FIT – ordena por razão valor/peso (pior custo-benefício primeiro)
def worst_fit(capacity: float, items: List[Tuple[float, float]]) -> Tuple[float, List[int]]:
    """
    Ordena itens pela razão valor/peso (crescente – pior primeiro) e insere.
    """
    ratios = []
    for i, (w, v) in enumerate(items):
        ratio = v / w if w > 0 else float('inf')
        ratios.append((ratio, i, w, v))
    ratios.sort()  # crescente = pior razão primeiro

    remaining = capacity
    total_value = 0.0
    selected = []
    for _, i, w, v in ratios:
        if w <= remaining:
            selected.append(i)
            remaining -= w
            total_value += v
    return total_value, selected


In [30]:
# 5. SUBIDA DE ENCOSTA (Hill Climbing)

def subida_Encosta(capacity: float, items: List[Tuple[float, float]],
                  max_iter: int = 1000, seed: int = 42) -> Tuple[float, List[int]]:
    """
    Subida de Encosta com reinício aleatório.
    Representação: vetor binário (0/1 por item).
    Vizinhança: flip de um bit (inserir ou remover item).
    """
    random.seed(seed)
    n = len(items)
    weights = [w for w, _ in items]
    values  = [v for _, v in items]

    def fitness(sol):
        total_w = sum(weights[i] * sol[i] for i in range(n))
        total_v = sum(values[i]  * sol[i] for i in range(n))
        return total_v if total_w <= capacity else -1e9

    def random_solution():
        sol = [0] * n
        remaining = capacity
        order = list(range(n))
        random.shuffle(order)
        for i in order:
            if weights[i] <= remaining:
                sol[i] = 1
                remaining -= weights[i]
        return sol

    best_sol = random_solution()
    best_val = fitness(best_sol)

    for _ in range(max_iter):
        current = best_sol[:]
        improved = False
        order = list(range(n))
        random.shuffle(order)
        for i in order:
            neighbor = current[:]
            neighbor[i] = 1 - neighbor[i]
            val = fitness(neighbor)
            if val > best_val:
                best_sol = neighbor
                best_val = val
                improved = True
                break  # aceita a primeira melhora
        if not improved:
            # Reinício aleatório se preso em ótimo local
            candidate = random_solution()
            candidate_val = fitness(candidate)
            if candidate_val > best_val:
                best_sol = candidate
                best_val = candidate_val

    selected = [i for i in range(n) if best_sol[i] == 1]
    return best_val, selected

In [31]:

def executarTudo(filepath: str):
    filename = os.path.basename(filepath)
    n, capacity, items = read_instance(filepath)

    print(f"\n{'═'*70}")
    print(f"  Arquivo : {filename}")
    print(f"  Itens   : {n}   │   Capacidade: {capacity}")
    print(f"{'═'*70}")
    print(f"  {'Algoritmo':<25} {'Valor':>12} {'Peso Usado':>12} {'Tempo (ms)':>12}")
    print(f"  {'─'*25} {'─'*12} {'─'*12} {'─'*12}")

    algorithms = [
        ("Exato (PD)",        exato_PD),
        ("First Fit",         first_fit),
        ("Best Fit",          best_fit),
        ("Worst Fit",         worst_fit),
        ("Subida de Encosta", subida_Encosta),
    ]

    resul = {}
    for name, func in algorithms:
        t0 = time.perf_counter()
        val, sel = func(capacity, items)
        elapsed = (time.perf_counter() - t0) * 1000

        peso_usado = sum(items[i][0] for i in sel)
        resul[name] = val
        print(f"  {name:<25} {val:>12.4f} {peso_usado:>12.4f} {elapsed:>12.4f}")

    # Verificar se heuristica atingiu o otimo
    exact_val = resul.get("Exato (PD)", 1)
    print(f"\n  {'Algoritmo':<25} {'Resultado':<14} {'Status'}")
    print(f"  {'─'*25} {'─'*14} {'─'*20}")
    for name, val in resul.items():
        if name == "Exato (PD)":
            status = "-- otimo exato --"
        elif val >= exact_val:
            status = "Atingiu o otimo"
        else:
            status = "Abaixo do otimo"
        print(f"  {name:<25} {val:>12.4f}   {status}")


def main():
    paths = sorted(
        os.path.join(PASTA, f)
        for f in os.listdir(PASTA)
        if os.path.isfile(os.path.join(PASTA, f))
    )
    
    print("\nPROBLEMA DA MOCHILA > Comparação de Algoritmos > Exato, First Fit, Best Fit, Worst Fit, Subida de Encosta.")
    
    for fpath in paths:
        executarTudo(fpath)

    print("\n" + "═"*70)
    print("  Execução concluída.")


if __name__ == "__main__":
    main()


PROBLEMA DA MOCHILA > Comparação de Algoritmos > Exato, First Fit, Best Fit, Worst Fit, Subida de Encosta.

══════════════════════════════════════════════════════════════════════
  Arquivo : f10_l-d_kp_20_879
  Itens   : 20   │   Capacidade: 879.0
══════════════════════════════════════════════════════════════════════
  Algoritmo                        Valor   Peso Usado   Tempo (ms)
  ───────────────────────── ──────────── ──────────── ────────────
  Exato (PD)                   1042.0000     879.0000    8875.6902
  First Fit                     836.0000     849.0000       0.0102
  Best Fit                     1027.0000     872.0000       0.0183
  Worst Fit                     668.0000     875.0000       0.0118
  Subida de Encosta            1042.0000     879.0000     265.5116

  Algoritmo                 Resultado      Status
  ───────────────────────── ────────────── ────────────────────
  Exato (PD)                   1042.0000   -- otimo exato --
  First Fit                     836